In [1]:
import pandas as pd
import h5py
import numpy as np

# USER INPUT

In [2]:

#Function to read user input from spreadsheet

def read_sample_data(identifier, file_path):
    # Read from spreadsheet
    
    # Open file
    had_to_copy = False
    try:
        df = pd.read_excel(file_path, sheet_name="Foglio1")
    except PermissionError:
        print('Error reading external file. \nIt might be a .xlsx open in another program (PermissionError). Trying to read from a temporary copy.\n')
        import subprocess
        import os
        subprocess.call(['powershell.exe', f'copy {file_path} nxellipsometry_from_woollam_tempfile'])
        had_to_copy = True
        df = pd.read_excel('nxellipsometry_from_woollam_tempfile', sheet_name="Foglio1")

    # Find the row that matches the identifier
    row = df[df['Identificatore'] == identifier]
    
    # Check that there is exactly one row matching the identifier and raise appropriate errors otherwise
    if row.empty:
        raise ValueError(f"Identifier '{identifier}' not found in any row of the file.")
    if len(row) != 1:
        raise ValueError(f"Identifier '{identifier}' is not unique in the file (found in {len(row)} rows).")

    # Extracting values from the row
    user_input = {
        'filepath': row['Filepath'].values[0],
        'outputfile': row['Output File'].values[0] if pd.notna(row['Output File'].values[0]) else "output.nxs", # assign a default value if no outputfile name is provided
        'title': row['Titolo'].values[0],
        'ellipsometry_date': row['Data Ellissometria'].values[0],
        'description': row['Descrizione'].values[0],
        'sample_name': row['Nome Campione'].values[0],
        'sample_backside_roughness': row['Backside Roughness'].values[0],
        'sample_ID': row['ID campione'].values[0],
        'sample_description': row['Descrizione Campione'].values[0],
        'sample_chemical_formula': row['Formula Chimica'].values[0],
        'sample_atoms': row['Atomi'].values[0],
        'sample_preparation_date': row['Data Preparazione'].values[0],
        'sample_layers': row['Layers'].values[0],
        'sample_substrate': row['Substrato'].values[0],
        'sample_ID': row['ID campione'].values[0],
    }

    if pd.isna(user_input['filepath']):
        raise ValueError("No path to file provided.")

    # Extracting user values from the Operatori worksheet
    Operatore = row['Operatore'].values[0]
    if not pd.isna(Operatore):
        if not had_to_copy:
            userdf = pd.read_excel(file_path, sheet_name="Operatori")
        else:
            userdf = pd.read_excel('nxellipsometry_from_woollam_tempfile', sheet_name="Operatori")
        
        userrow = userdf[userdf['Operatore'] == Operatore]
        # Check that there is exactly one match
        if userrow.empty:
            raise ValueError(f"'{Operatore}' not found in the Operatori worksheet.")
        if len(userrow) != 1:
            raise ValueError(f"'{Operatore}' not unique (found {len(userrow)} rows).")

        user_info_dict = {
            'user_name': userrow['Nome Completo'].values[0],
            'user_role': userrow['Ruolo'].values[0],
            'user_affiliation': userrow['Affiliazione'].values[0],
            'user_address': userrow['Indirizzo'].values[0],
            'user_telephone_number': userrow['Telefono'].values[0],
            'user_email': userrow['Email'].values[0],
            'user_ORCID': userrow['ORCID'].values[0],
        }

        user_input.update(user_info_dict)
    
    # remove empty values, they shouldn't be transcribed to the .nxs file
    user_input = {key: value for key, value in user_input.items() if not pd.isna(value)}

    # if we had to copy the file, delete the temporary copy
    if had_to_copy:
        os.remove('nxellipsometry_from_woollam_tempfile')

    return user_input

In [3]:
# User input from file

identifier = 'esempio conversione'
spreadsheet_filepath = 'ELLIPSOMETER_CNR-IMM@CT_ELN.ods'
user_input = read_sample_data(identifier, spreadsheet_filepath)

In [4]:
# Direct user input

##File paths
#user_input['filepath'] = 'test file.dat'
#user_input['outputfile'] = 'output.nxs'

##General
#user_input['title'] = 'corning 1737 glass slide'
#user_input['ellipsometry_date'] = '2024-10-23' # format yyyy-mm-dd, or ISO 8601
#user_input['description'] = 'Ellipsometry of glass substrate Corning 1737'

##Sample
#user_input['sample_name'] = 'corning 1737'
#user_input['sample_backside_roughness'] = True
#user_input['sample_ID'] = 'exampleID'
#user_input['sample_description'] = 'Microscope slide made of Corning 1737 glass'
#user_input['sample_chemical_formula'] = 'O2Si' #Hill notation, see wikipedia.
#user_input['sample_atoms'] = ['O', 'Si']
#user_input['sample_preparation_date'] = '2024-10-23' # yyy-mm-dd, or ISO 8601
#user_input['sample_layers'] = 'SiO2/glass' # In order starting from the surface layer surface/layer1/.../substrate
#user_input['sample_substrate'] = 'glass'

##User
#user_input['user_name'] = 'Donald Duck'
#user_input['user_role'] = 'Ricercatore III livello'
#user_input['user_affiliation'] = 'Istituto per la Microelettronica e i Microsistemi'
#user_input['user_address'] = '95121 - Catania, Italy - Strada VIII, 5'
#user_input['user_telephone_number'] = 'N/A'
#user_input['user_email'] = 'TheDonald@papermail.it'
#user_input['user_ORCID'] = '0009-0003-6881-8642'

# 1: Reading ellipsometry .dat file

### 1.1 Reading metadata from the header

In [5]:
# List of known parameters in VASEmethod; also checks if there are unknown parameters.
# Order is IMPORTANT, because some key strings are in other key strings (e.g. 'Revs' in 'MaxRevs')

known_VASEmethod_parameters=["EllipsometerType",
                  "Isotropic+Depolarization",
                  "Isotropic",
                  "AutoRetarder",
                  "TrackPol",
                  "Pol",
                  "MaxRevs",
                  "Revs",
                  "I_Threshold",
                  "I_MinUsable",
                  "AutoSlit",
                  "WVASE",
                  "HardVer",
                  ]
known_VASEmethod_parameters+=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]


def check_unknown_parameters(filepath, known_VASEmethod_parameters):
    # Rough but works

    with open(filepath) as file:
        file_string=file.read()
        try:
            begin_index=file_string.index("VASEmethod[")+len("VASEmethod[")
            end_index=file_string[begin_index:].index("]")+begin_index
            unknown_parameters=file_string[begin_index:end_index]
            
            #remove all known parameters
            for p in known_VASEmethod_parameters:
                if p in unknown_parameters:
                    try: unknown_parameters = unknown_parameters[:unknown_parameters.find(p)]+unknown_parameters[unknown_parameters.index(",", unknown_parameters.find(p))+1:]
                    except ValueError: unknown_parameters = unknown_parameters[:unknown_parameters.find(p)]
            
            #check if unknown_parameters is now only spaces or empty
            if not "".join(unknown_parameters.split()) == "":
                print("Unknown parameters in", filepath, ":")
                print(unknown_parameters)
            else:
                print("No unknown parameters in", filepath) #unnecessary
        except ValueError:
            print("----------------WARNING!----------------\nlikely no VASEmethod in",filepath)
            print("here are the first few lines in", filepath ,":")
            i=0
            for line in file_string.split("\n",7)[0:7]:
                if len(line)>80: print(i, "\t", line[0:80], "...truncated...")
                else: print(i, "\t", line)
                i+=1
            print("...\n----------------------------------------")

check_unknown_parameters(user_input['filepath'], known_VASEmethod_parameters)


No unknown parameters in ELLIPSOMETER_CNR-IMM@CT_input.dat


In [6]:
#Reading the VASEmethod metadata:

#General procedure:
#1. find the VASEmethod[...] string in the file and extract the part inside square brackets
#2. extract the parameters into a dictionary, one by one with ad-hoc procedures.


#1. read
with open(user_input['filepath']) as file:
    file_string=file.read()
    begin_index=file_string.index("VASEmethod[")+len("VASEmethod[")
    end_index=file_string[begin_index:].index("]")+begin_index
    VASEmethod_string=file_string[begin_index:end_index]

#2.
#-one by one, look for each known parameter and assign the value in a dictionary
#-parameters are stored as strings
#-some consistency checks are also performed

# Initialize empty dictionary
VASEmethod={}

# Ellipsometer Type
if "EllipsometerType" in VASEmethod_string: 
    VASEmethod['EllipsometerType']=VASEmethod_string[VASEmethod_string.find("EllipsometerType")+len("EllipsometerType=")]
    if VASEmethod['EllipsometerType'] != "5":
        print("-----------WARNING-----------\n Unknown value of EllipsometerType, check the metadata manually for code improvements")
else:
    raise ValueError("no EllipsometerType detected, unknown meaning")


# isotropic, isotropic+depolarization, and similar
# these MUST be checked in the correct order, as for instance Isotropic+Depolarization is also detected as Isotropic
sample_types=["Isotropic",
              "Isotropic+Depolarization",
              # Other possible sample_type according to the manual:
              #"Slightly Anisotropic",
              #"Highly Anisotropic",
              #"Anisotropic (no regression)",
              #"Mueller matrix",
              ]
for s in sample_types:
    if s in VASEmethod_string:
        VASEmethod['sample_type'] = s
if not 'sample_type' in VASEmethod:
    print("-----------WARNING-----------\n unrecognized sample type in VASEmethod string")

#AutoRetarder, value 1 or 0 meaning on or off, or absent.
if "AutoRetarder" in VASEmethod_string: 
    VASEmethod['AutoRetarder']=VASEmethod_string[VASEmethod_string.find("AutoRetarder")+len("AutoRetarder=")]
    #we currently don't know what the metadata looks like with AutoRetarder=0, so:
    if VASEmethod['AutoRetarder'] == '0':
        print("-----------Attention-----------\n AutoRetarder=0, inspect the metadata for potential code improvements")
    if VASEmethod['AutoRetarder'] not in ['0', '1']:
        raise ValueError("unexpected value of AutoRetarder, unknown meaning")
        
#Software, expected "WVASE"
if "WVASE" in VASEmethod_string:
    VASEmethod['software']="WVASE"
    try:
        VASEmethod['software_version']=VASEmethod_string[VASEmethod_string.find("WVASE")+len("WVASE="):VASEmethod_string.index(",", VASEmethod_string.find("WVASE"))]
    except ValueError:
        VASEmethod['software_version']=VASEmethod_string[VASEmethod_string.find("WVASE")+len("WVASE="):]
if VASEmethod.get('software') != "WVASE":
    print("-----------Attention-----------\n unexpected or missing software, check metadata")
elif VASEmethod.get('software_version') != "3.810":
    print("-----------Attention-----------\n unexpected software version, check metadata")

#HardVer, expected 6.256
if "HardVer" in VASEmethod_string:
    try:
        VASEmethod['HardVer']=VASEmethod_string[VASEmethod_string.find("HardVer")+len("HardVer="):VASEmethod_string.index(",", VASEmethod_string.find("HardVer"))]
    except ValueError:
        VASEmethod['HardVer']=VASEmethod_string[VASEmethod_string.find("HardVer")+len("HardVer="):]
if VASEmethod['HardVer'] != "6.256":
    print("-----------Attention-----------\n unexpected or missing value of HardVer, check metadata")

#TrackPol, expected 0 or 1
if "TrackPol" in VASEmethod_string: VASEmethod['TrackPol']=VASEmethod_string[VASEmethod_string.find("TrackPol")+len("TrackPol=")]
if VASEmethod.get('TrackPol') != "1":
    print("TrackPol value not previously encountered, check metadata for potential code improvements")
    #TrackPol has always been 1 in all test data. If data with TrackPol 0 are used, take the chance to inspect the metadata closer.

#Pol
#Workaround to avoid conflict with "TrackPol", check if " Pol" is present instead of "Pol"; see comments on "Revs"
if " Pol" in VASEmethod_string:
    try:
        VASEmethod['Pol']=VASEmethod_string[VASEmethod_string.find(" Pol")+len(" Pol"):VASEmethod_string.index(",", VASEmethod_string.find(" Pol"))]
    except ValueError:
        VASEmethod['Pol']=VASEmethod_string[VASEmethod_string.find(" Pol")+len(" Pol"):]
if VASEmethod.get('Pol') not in [">=0", "<0"]:
    print("-----------Attention-----------\n Unexpected value of Pol, check the metadata for potential code improvements")

#MaxRevs
if "MaxRevs" in VASEmethod_string:
    try:
        VASEmethod['MaxRevs']=VASEmethod_string[VASEmethod_string.find("MaxRevs")+len("MaxRevs="):VASEmethod_string.index(",", VASEmethod_string.find("MaxRevs"))]
    except ValueError:
        VASEmethod['MaxRevs']=VASEmethod_string[VASEmethod_string.find("MaxRevs")+len("MaxRevs="):]

#Revs
#"Revs" could detect "MaxRevs", as "Revs" in "MaxRevs"; workaround with a space at beginning " Revs" should work
#proper way would be to user regex, but if it ain't broke don't fix it.
if " Revs" in VASEmethod_string:
    try:
        VASEmethod['Revs']=VASEmethod_string[VASEmethod_string.find(" Revs")+len(" Revs="):VASEmethod_string.index(",", VASEmethod_string.find(" Revs"))]
    except ValueError:
        VASEmethod['Revs']=VASEmethod_string[VASEmethod_string.find(" Revs")+len(" Revs="):]

#I_Threshold
if "I_Threshold" in VASEmethod_string:
    try:
        VASEmethod['I_Threshold']=VASEmethod_string[VASEmethod_string.find("I_Threshold")+len("I_Threshold="):VASEmethod_string.index(",", VASEmethod_string.find("I_Threshold"))]
    except ValueError:
        VASEmethod['I_Threshold']=VASEmethod_string[VASEmethod_string.find("I_Threshold")+len("I_Threshold="):]

#I_MinUsable
if "I_MinUsable" in VASEmethod_string:
    try:
        VASEmethod['I_MinUsable']=VASEmethod_string[VASEmethod_string.find("I_MinUsable")+len("I_MinUsable="):VASEmethod_string.index(",", VASEmethod_string.find("I_MinUsable"))]
    except ValueError:
        VASEmethod['I_MinUsable']=VASEmethod_string[VASEmethod_string.find("I_MinUsable")+len("I_MinUsable="):]

#AutoSlit
if "AutoSlit" in VASEmethod_string:
    try:
        VASEmethod['AutoSlit']=VASEmethod_string[VASEmethod_string.find("AutoSlit")+len("AutoSlit="):VASEmethod_string.index(",", VASEmethod_string.find("AutoSlit"))]
    except ValueError:
        VASEmethod['AutoSlit']=VASEmethod_string[VASEmethod_string.find("AutoSlit")+len("AutoSlit="):]
if VASEmethod.get('AutoSlit') != "1700":
    print("-----------Attention-----------\n Unexpected value of AutoSlit, check metadata for potential code improvements")

#Measurement date stamp
for Day in ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]:
    if Day in VASEmethod_string:
        try:
            VASEmethod['measurement_date']=VASEmethod_string[VASEmethod_string.find(Day):VASEmethod_string.index(",", VASEmethod_string.find(Day))]
        except ValueError:
            VASEmethod['measurement_date']=VASEmethod_string[VASEmethod_string.find(Day):]


### 1.2 Reading the actual data

In [7]:
#Read the actual data 
#into a pandas DataFrame

#detect header length
with open(user_input['filepath']) as file:
    for idx, line in enumerate(file):
        columns = line.strip().split('\t')
        if len(columns) > 1:         
            header_length = idx
            break
if header_length == 0:
    raise Exception("zero-length header; The script should not be able to get to this part without failing on the VASEmethod checks, so something else is wrong.")

#read the data to a pandas dataframe
df=pd.read_csv(user_input['filepath'],  delimiter="\t", skiprows=header_length, header=None)


#read the units
#Units can be any of: "Angstrom", "nm", "eV", "um", "1/cm".
units_list = ["Angstrom", 
            "nm",
            "eV",
            "um",
            "1/cm",
            ]

with open(user_input['filepath']) as file:
    for i, line in enumerate(file):
        if i==(header_length-1):
            units=line[0:-1]
            break
        elif i>(header_length-1):
            raise Exception
    if units not in units_list:
        print("-----------Attention-----------\nUnexpected spectral units, defaulting to Angstrom.")
        units="Angstrom"
        print(f"Please check data file manually.\nLine that is expected to have the spectral units:\n{line}")

    

In [8]:

#split dataframe containing all data into individual data type dataframes

#depolarization data DataFrame
depolarization_data=df[(df[0].str.contains("dpol"))]
depolarization_data=depolarization_data.reset_index(drop=True).dropna(axis=1, how="all") 
df = df[~(df[0].str.contains("dpol"))]

#transmission data DataFrame
transmission_data=df[df[0].isin(["pT", "pTb", "pTr"])]
transmission_data=transmission_data.reset_index(drop=True).dropna(axis=1, how="all")
df = df[~(df[0].isin(["pT", "pTb", "pTr"]))]

ellipsometric_data=df.copy(deep=True)
#Convert first column from strings to numbers.
ellipsometric_data[0]=pd.to_numeric(ellipsometric_data[0], errors="raise")

# 2: Writing to NeXus format

In [9]:
#Write metadata t NeXus file

try: f.close()
except NameError: pass

f=h5py.File(user_input['outputfile'],"w")

#ENTRY
f.create_group("/entry")
f["/entry"].attrs["NX_class"]="NXentry"

f["/entry/definition"]="NXellipsometry"
f["/entry/definition"].attrs["version"]="2025-07-24"
f["/entry/definition"].attrs["url"]="https://github.com/nexusformat/definitions/blob/023c5f0889cce9e37c8028e941e04ebba126273e/applications/NXellipsometry.nxdl.xml"

if 'title' in user_input:
    f["/entry/title"]=user_input['title']

f["/entry/experiment_type"]="ellipsometry"
f["/entry/ellipsometry_experiment_type"]="NIR-Vis-UV spectroscopic ellipsometry"

#extra /ENTRY fields from NXoptical_spectroscopy
if "ellipsometry_date" in user_input:
    f["/entry/start_time"]=str(user_input['ellipsometry_date'])[:10]
if "description" in user_input:
    f["/entry/experiment_description"]=user_input['description']

#extra /ENTRY fields from NXentry
f["/entry/experiment_location"] = "Catania, Italy"
f["/entry/experiment_institution"]="Istituto per la Microelettronica e i Microsistemi"
f["/entry/experiment_facility"]="Spectroscopic Ellipsometry Lab"

#INSTRUMENT
f.create_group("/entry/instrument")
f["/entry/instrument"].attrs["NX_class"]="NXinstrument"
if VASEmethod['AutoRetarder'] == '1':
    f["/entry/instrument/ellipsometer_type"]="rotating analyzer with polarizer compensator"
else:
    raise Exception("AutoRetarder not equal to 1; understand what it means and update the code.\nProbably, f[\"/entry/instrument/ellipsometer_type\"] should be = \"rotating analyzer\"")

f.create_group("/entry/instrument/rotating_element")
f["/entry/instrument/rotating_element"].attrs["NX_class"] = "NXwaveplate"
f["/entry/instrument/rotating_element/rotating_element_type"] = "analyzer (detector side)"
if "MaxRevs" in VASEmethod:
    f["/entry/instrument/rotating_element/max_revolutions"] = int(VASEmethod['MaxRevs'])
    if "Revs" in VASEmethod: f["/entry/instrument/rotating_element/revolutions"] = int(VASEmethod['Revs'])
#FixedRevs actually never found in test files
elif "FixedRevs" in VASEmethod:
    f["/entry/instrument/rotating_element/fixed_revolutions"] = int(VASEmethod['FixedRevs'])
    print("FixedRevs has been found in the metadata. Please inspect the data file and consider updating the script.")

#extra /entry/instrument fields from NXoptical_spectroscopy
f["/entry/instrument/angle_reference_frame"]="sample-normal centered"
#/instrument/angle_of_incidence e /angle_of_detection are assigned later, among actual data
f.create_group("/entry/instrument/device_information")
f["/entry/instrument/device_information"].attrs["NX_class"]="NXfabrication"
f["/entry/instrument/device_information/vendor"]="J.A. Woollam"
f["/entry/instrument/device_information/model"]="VASE®"
if 'HardVer' in VASEmethod:
    f["/entry/instrument/device_information/model"].attrs["version"]=VASEmethod.get('HardVer')
else:
    f["/entry/instrument/device_information/model"].attrs["version"]="6.256"
    print("-----------attention-----------\n HardVer not found in VASEmethod metadata, using default value 6.256")
#f["/entry/instrument/device_information/construction_year"]="placeholder"
#f["/entry/instrument/device_information/serial_number"]="placeholder"

#extra /entry/instrument fields from NXinstrument

#SAMPLE
f.create_group("/entry/sample")
f["/entry/sample"].attrs["NX_class"]="NXsample"
if "sample_backside_roughness" in user_input:
    f["/entry/sample/backside_roughness"] = user_input['sample_backside_roughness']

#extra /entry/sample fields from NXoptical_spectroscopy
#note: none of these actually NEED to be a given format, as they are not required in NXellipsometry
#but there are still conventions for chemical formulas which are not trivial to match
#so it would be appropriate to have specific functions to check and parse the format of e.g. a chemical formula
if "sample_name" in user_input:
    f["/entry/sample/name"]=user_input['sample_name']
if "sample_id" in user_input:
    f["/entry/sample/id"]=user_input['sample_ID']
if "sample_description" in user_input:
    f["/entry/sample/description"]=user_input['sample_description']
if "sample_chemical_formula" in user_input:
    f["/entry/sample/chemical_formula"]=user_input['sample_chemical_formula']
if "sample_atoms" in user_input:
    f["/entry/sample/atom_types"]=user_input['sample_atoms']
if "preparation_date" in user_input:
    f["/entry/sample/preparation_date"]=user_input['preparation_date']
if "sample_layers" in user_input:
    f["/entry/sample/layer_structure"]=user_input['sample_layers']
if "substrate" in user_input:
    f["/entry/sample/substrate"]=user_input['substrate']

#extra groups not in NXellipsometry
#USER:NXuser
if "user_name" in user_input:
    f.create_group("/entry/user")
    f["/entry/user"].attrs["NX_class"]="NXuser"
    f["/entry/user/name"]=user_input['user_name']
    if "user_role" in user_input:
        f["/entry/user/role"]=user_input['user_role']
    if "user_affiliation" in user_input:
        f["/entry/user/affiliation"]=user_input['user_affiliation']
    if "user_address" in user_input:
        f["/entry/user/address"]=user_input['user_address']
    if "user_telephone_number" in user_input:
        f["/entry/user/telephone_number"]=user_input['user_telephone_number']
    if "user_email" in user_input:
        f["/entry/user/email"]=user_input['user_email']
    if "user_ORCID" in user_input:
        f.create_group("/entry/user/identifier")
        f["/entry/user/identifier"].attrs["NX_class"]="NXidentifier"
        f["/entry/user/identifier/service"]="orcid"
        f["/entry/user/identifier/identifier"]=user_input['user_ORCID']
        f["/entry/user/identifier/is_persistent"]=True

In [10]:
#Write data to NeXus file

#angles
f["/entry/instrument/angle_of_incidence"]=ellipsometric_data[1].unique()
f["/entry/instrument/angle_of_incidence"].attrs["units"]="deg"
#angle of detection is equal to angle of incidence in ellipsometry
#keeping as placeholder
#f["/entry/instrument/angle_of_detection"]=ellipsometric_data[1].unique()
#f["/entry/instrument/angle_of_detection"].attrs["units"]="deg"

#ellipsometry data, assuming that they are Psi/Delta and exist
f.create_group("/entry/data_collection")
f["/entry/data_collection"].attrs["NX_class"]="NXdata"
f["/entry/data_collection/data_identifier"]=1
f["/entry/data_collection/data_type"]="Psi/Delta"

f.create_group("/entry/data_collection/data_software")
f["/entry/data_collection/data_software"].attrs["NX_class"]="NXprogram"
f["/entry/data_collection/data_software/program"]=VASEmethod['software']
f["/entry/data_collection/data_software/version"]=VASEmethod['software_version']

#auxiliary
wavelengths=ellipsometric_data[0].unique()
wavelengths.sort()

#NAME_spectrum
if units in ["Angstrom", "nm", "um"]:
    NAME_spectrum="wavelength_spectrum"
elif units in ["eV"]:
    NAME_spectrum="energy_spectrum"
elif units in ["1/cm"]:
    NAME_spectrum="wavevenumber_spectrum"
else:
    raise ValueError("unrecognized spectral units") 

f["/entry/data_collection/"+NAME_spectrum]=wavelengths
f["/entry/data_collection/"+NAME_spectrum].attrs["units"]=units

def fill_missing_rows(ellipsometry_dataframe, wavelengths):
    edf=ellipsometry_dataframe
    angles=edf[1].unique()
    for angle in angles:
        angle_wavelengths=edf[edf[1]==angle][0].unique()
        for wvl in wavelengths:
            if wvl not in angle_wavelengths:
                edf=edf._append(pd.Series([wvl, angle]), ignore_index=True)
    edf=edf.sort_values([1,0]).reset_index(drop=True)
    return(edf)
ellipsometric_data=fill_missing_rows(ellipsometric_data,wavelengths)

angles=ellipsometric_data[1].unique()
ell_data_matrix=np.ndarray(shape=[len(angles), 2, len(wavelengths)])
for i in range(len(angles)):
    for j in range(2):
        ell_data_matrix[i,j]=ellipsometric_data[ellipsometric_data[1]==angles[i]][2+j].values

ell_error_matrix=np.ndarray(shape=[len(angles), 2, len(wavelengths)])
for i in range(len(angles)):
    for j in range(2):
        ell_error_matrix[i,j]=ellipsometric_data[ellipsometric_data[1]==angles[i]][4+j].values

f["/entry/data_collection/measured_data"]=ell_data_matrix
f["/entry/data_collection/measured_data"].attrs["units"]="deg"
f["/entry/data_collection/measured_data_errors"]=ell_error_matrix
f["/entry/data_collection/measured_data_errors"].attrs["units"]="deg"

def fill_missing_rows_dpol(dpol_dataframe, wavelengths):
    ddf=dpol_dataframe
    angles=ddf[2].unique()
    for angle in angles:
        angle_wavelengths=ddf[ddf[2]==angle][1].unique()
        for wvl in wavelengths:
            if wvl not in angle_wavelengths:
                ddf=ddf._append(pd.Series(["dpolE", wvl, angle]), ignore_index=True)

    ddf=ddf.sort_values([2,1]).reset_index(drop=True)
    return(ddf)

depolarization_data=fill_missing_rows_dpol(depolarization_data,wavelengths)

dpol_data_matrix=np.ndarray(shape=[len(angles), 1, len(wavelengths)])
for i in range(len(angles)):
        dpol_data_matrix[i,0]=depolarization_data[depolarization_data[2]==angles[i]][3].values

f.create_group("/entry/derived_parameters")
f["/entry/derived_parameters"].attrs["NX_class"]="NXprocess"
f["/entry/derived_parameters/depolarization"]=dpol_data_matrix
f["/entry/derived_parameters/depolarization"].attrs["units"]="unitless"



In [11]:
#create fields for default plot in hdf5 visualization tools

for angle in angles:
    anglestr=str(int(round(angle)))+"deg"
    f["/entry/data_collection/Psi_"+anglestr]=ellipsometric_data[ellipsometric_data[1]==angle][2]
    f["/entry/data_collection/Delta_"+anglestr]=ellipsometric_data[ellipsometric_data[1]==angle][3]
    f["/entry/data_collection/Psi_"+anglestr+"_errors"]=ellipsometric_data[ellipsometric_data[1]==angle][4]
    f["/entry/data_collection/Delta_"+anglestr+"_errors"]=ellipsometric_data[ellipsometric_data[1]==angle][5]
    f["/entry/data_collection/Psi_"+anglestr].attrs["units"]="deg"
    f["/entry/data_collection/Delta_"+anglestr].attrs["units"]="deg"
    f["/entry/data_collection/Psi_"+anglestr+"_errors"].attrs["units"]="deg"
    f["/entry/data_collection/Delta_"+anglestr+"_errors"].attrs["units"]="deg"
f.attrs["default"]="entry"
f["/entry"].attrs["default"]="data_collection"
f["/entry/data_collection"].attrs["axes"]=NAME_spectrum
f["/entry/data_collection"].attrs["signal"]="Psi_"+str(int(round(angles[0])))+"deg"
aux_signals=[]
for fieldname in ["Psi", "Delta"]:
    for angle in angles:
        anglestr=str(int(round(angle)))+"deg"
        aux_signals+=[fieldname+"_"+anglestr]
aux_signals.pop(0)
f["/entry/data_collection"].attrs["auxiliary_signals"]=aux_signals

In [12]:
#Extra data_collection group for optional transmittance data

if not transmission_data.empty:
    f.create_group("/entry/data_collection_2")
    f["/entry/data_collection_2"].attrs["NX_class"]="NXdata"
    f["/entry/data_collection_2/data_identifier"]=2
    f.create_group("/entry/data_collection_2/data_software")
    f["/entry/data_collection_2/data_software"].attrs["NX_class"]="NXprogram"
    f["/entry/data_collection_2/data_software/program"]=VASEmethod['software']
    f["/entry/data_collection_2/data_software/version"]=VASEmethod['software_version']
    f["/entry/data_collection_2/data_type"]="transmittance"

    #NAME_spectrum
    f["/entry/data_collection_2/"+NAME_spectrum]=transmission_data[1].values

    if transmission_data[2].unique()!=0:
        print("---------Attention!---------\nnon-zero angle of incidence found in transmission data")

    #measured_data
    f["/entry/data_collection_2/measured_data"]=transmission_data[3].values
    f["/entry/data_collection_2/measured_data"].attrs["units"]="unitless"
    #errors
    f["/entry/data_collection_2/measured_data_errors"]=transmission_data[4].values
    f["/entry/data_collection_2/measured_data_errors"].attrs["units"]="unitless"

    #can I set a default plot inside this group?
    f["/entry/data_collection_2"].attrs["axes"]=NAME_spectrum
    f["/entry/data_collection_2"].attrs["signal"]="measured_data"


In [13]:
f.close()
